In [2]:
%pip install unsloth

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
%pip install --upgrade unsloth-zoo
%pip install --upgrade unsloth

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
%pip install ipywidgets -U

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 22.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ipywidgets]4 [ipywidgets]widgets]

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
!jupyter nbextension enable --py widgetsnbextension

In [1]:
!nvidia-smi

Thu Oct  2 13:35:39 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.261.03             Driver Version: 535.261.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  | 00000000:8C:00.0 Off |                    0 |
| N/A   28C    P0              71W / 500W |      9MiB / 81920MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

## Data

In [2]:
import json
import pandas as pd
import numpy as np

import glob
import os

# from tqdm.autonotebook import tqmd

In [3]:
# df = pd.read_csv("data/data_latest.csv")

# df

### creating dataset

In [4]:
from unsloth import FastModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2025-10-02 13:37:30.057522: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-02 13:37:35.710559: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
max_seq_length = 4096

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-270m-it",
    max_seq_length = max_seq_length, # Choose any for long context!
    # load_in_4bit = False,  # 4 bit quantization to reduce memory
    # load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = True, # [NEW!] We have full finetuning now!
    token = "token", # use one if using gated models
)

Unsloth: You selected full finetuning support, but 4bit / 8bit is enabled - disabling LoRA / QLoRA.
==((====))==  Unsloth 2025.9.10: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.325 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.


In [6]:
# model = FastModel.get_peft_model(
#     model,
#     r = 128, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
#     target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
#                       "gate_proj", "up_proj", "down_proj",],
#     lora_alpha = 128,
#     lora_dropout = 0, # Supports any, but = 0 is optimized
#     bias = "none",    # Supports any, but = "none" is optimized
#     # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
#     use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
#     random_state = 3407,
#     use_rslora = False,  # We support rank stabilized LoRA
#     loftq_config = None, # And LoftQ
# )

In [7]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma3",
)

In [8]:
# print(df["review_text"][df["review_id"] == 573263].values)

In [9]:
# synth_data

In [10]:
# {int(synth_data[i]["id"]) : synth_data[i]["topic_sentiment_pairs"] for i in range(10)}

In [15]:
SYSTEM_PROMPT = """\
Проанализируй отзыв клиента Газпромбанка (ГПБ) и определи:
1. Упоминаемые тем ("topic") из списка допустимых тем;
2. Тональность ("sentiment") для каждой темы: positive/negative/neutral.

ПРАВИЛА:
- Тональность: `neutral` указывается тогда, когда тема упомянута как факт, без эмоциональной окраски.
- Не выдумывай темы: Если в отзыве нет явного упоминания продукта или услуги, не включай его.
- Если темы нет: Если невозможно определить ни одну тему, верни пустой массив [].
- Темы: Используй ТОЛЬКО следующий список тем и подтем. Не добавляй новые темы.
- Уникальность тем: Одна тема может встречаться только один раз.

ДОПУСТИМЫЕ ТЕМЫ:
- Офисное обслуживание (обслуживание в отделениях банка)
- Дистанционное обслуживание (звонки, чаты, онлайн-консультации и подобное)
- Банкоматы
- Курьерская доставка карт
- Обмен валют
- Дебетовые карты (включая подтемы: Денежные переводы, Карта UnionPay, Умная дебетовая карта «Мир», Премиальная карта Mir Supreme)
- Кредитные карты (включая подтемы: Кредитная карта 180 дней Премиум)
- Кредиты (включая подтемы: Кредит наличными, Кредит наличными под залог недвижимости, Кредит под залог автомобиля)
- Рефинансирование/Реструктуризация (включая подтемы: Рефинансирование кредитов, Реструктуризация кредитов, Рефинансирование ипотеки, Реструктуризация ипотеки)
- Автокредиты
- Ипотека
- Страховые и сервисные продукты
- Вклады (включая подтемы: Вклад «Копить», Вклад «В Плюсе», Вклад «Новые деньги»)
- Накопительные счета (включая подтемы: Накопительный счёт «Ежедневная выгода», Накопительный счёт «Ежедневный процент», Накопительный счёт «Премиум»)
- Акции и бонусы (включая подтемы: Газпром Бонус, Газпромбанк Привилегии, Кэшбэк, Акции, Программы лояльности)
- Газпромбанк Премиум (включая подтемы: Персональный менеджер, Консьерж-сервис, Премиальное обслуживание)
- Мобильное приложение
- Другие услуги банка (включая подтемы: Газпромбанк Travel (покупка авиабилетов/отелей), Gazprom Pay (оплата телефоном), GorodPay (оплата общественного транспорта), Инвестиционные продукты, Брокерские услуги, Депозитарные услуги, Аренда сейфовых ячеек)

Примеры:
Отзыв: "В отделении грубо обслужили, но мобильное приложение удобное"
[
{"topic": "Офисное обслуживание", "sentiment": "negative"},
{"topic": "Мобильное приложение", "sentiment": "positive"}
]

Отзыв: "Курьер не пришёл на встречу. По телефону не смогли помочь."
[
{"topic": "Курьерская доставка карт", "sentiment": "negative"},
{"topic": "Дистанционное обслуживание", "sentiment": "negative"}
]

Отзыв: "Оформил Премиальную карту Mir Supreme через приложение"
[
{"topic": "Премиальная карта Mir Supreme", "sentiment": "neutral"},
{"topic": "Дебетовые карты", "sentiment": "neutral"},
{"topic": "Газпромбанк Премиум", "sentiment": "neutral"},
{"topic": "Мобильное приложение", "sentiment": "neutral"}
]

Отзыв: "Пользуюсь Газпромбанк Travel для бронирования отелей и Gazprom Pay для оплаты"
[
{"topic": "Другие услуги банка", "sentiment": "neutral"},
{"topic": "Газпромбанк Travel", "sentiment": "neutral"},
{"topic": "Gazprom Pay", "sentiment": "neutral"}
]

Проанализируй следующий отзыв:
"""

In [16]:
# Сдеалть разделение на train/test по дате

from datasets import Dataset

topics_sentiments_json = "dataset_v2.json"
# original_reviews_csv = "mount/data/data_latest.csv"

with open(topics_sentiments_json) as f:
    topics_sentiments_full = json.load(f)
    
# topics_sentiments_pair = {
#     int(topics_sentiments_full[i]["id"]) : topics_sentiments_full[i]["topic_sentiment_pairs"] 
#     for i in range(len(topics_sentiments_full))
# }

# original_reviews_df = pd.read_csv(original_reviews_csv)


user_prompts = []
assistant_answers = []

for i in range(len(topics_sentiments_full)):
    # original_review_series = original_reviews_df.iloc[i]
    # review_id = original_review_series["review_id"]

    # user_prompt = original_review_series["review_text"]
    
    user_prompt = topics_sentiments_full[i]["review_text"]

    assistant_answer = str(topics_sentiments_full[i]["topic_sentiment_pairs"])
    
    user_prompts.append(user_prompt)
    assistant_answers.append(assistant_answer)
    

dataset_dict = {"user_prompt" : user_prompts, "assistant_answer" : assistant_answers}

dataset = Dataset.from_dict(dataset_dict)

In [17]:
dataset = dataset.train_test_split(test_size=0.1, shuffle=True)

dataset_train = dataset["train"]
dataset_test = dataset["test"]

In [18]:
dataset_train[100]

{'user_prompt': 'Здравствуйте. Спасибо большое Огаджанян Татьяне за оказанную услугу - доставка карты. Очень хорошая девочка! Звонила утром, представилась, договорились о встрече. За 20 минут до приезда - дополнительный звонок.  Всё хорошо объяняет, четко по делу. Помогла определиться с выбором вклада! Не отказала в просьбе, хотя очень спешила - канун праздника, а ещё много клиентов. Большая молодец! Спасибо 🙏!',
 'assistant_answer': "[{'topic': 'Курьерская доставка карт', 'sentiment': 'positive'}, {'topic': 'Вклады', 'sentiment': 'positive'}]"}

In [19]:
def convert_to_chatml(example):
    return {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["user_prompt"]},
            {"role": "assistant", "content": example["assistant_answer"]}
        ]
    }

dataset_train = dataset_train.map(
    convert_to_chatml
)

dataset_test = dataset_test.map(
    convert_to_chatml
)

Map: 100%|██████████| 1155/1155 [00:00<00:00, 9868.09 examples/s] 


In [20]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts, }

dataset_train = dataset_train.map(formatting_prompts_func, batched = True)

dataset_test = dataset_test.map(formatting_prompts_func, batched = True)

Map: 100%|██████████| 1155/1155 [00:00<00:00, 7626.45 examples/s]


## training

In [21]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    eval_dataset = dataset_test, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 2, # Set this for 1 full training run.
        # max_steps = 100,
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.015,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir="gemma-3-270m-it-revews-fine-tune-v3",
        report_to = "none", # Use this for WandB etc
        dataset_num_proc=0,
        do_eval=True,
        eval_strategy="steps",
        eval_steps=0.2,
    ),
)

Unsloth: Tokenizing ["text"]: 100%|██████████| 1155/1155 [00:00<00:00, 1939.51 examples/s]


In [22]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
    num_proc=0,
)

Map: 100%|██████████| 1155/1155 [00:01<00:00, 1060.30 examples/s]


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,389 | Num Epochs = 2 | Total steps = 650
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 268,098,176 of 268,098,176 (100.00% trained)
  0%|          | 0/650 [00:00<?, ?it/s]

Unsloth: Will smartly offload gradients to save VRAM!


  8%|▊         | 50/650 [02:56<27:43,  2.77s/it] 

{'loss': 0.2563, 'grad_norm': 5.1875, 'learning_rate': 1.863565891472868e-05, 'epoch': 0.15}


 15%|█▌        | 100/650 [05:17<24:43,  2.70s/it]

...

 62%|██████▏   | 400/650 [22:20<13:43,  3.29s/it]  

{'loss': 0.0969, 'grad_norm': 6.84375, 'learning_rate': 7.782945736434108e-06, 'epoch': 1.23}


 69%|██████▉   | 450/650 [24:42<09:34,  2.87s/it]

{'loss': 0.0957, 'grad_norm': 4.8125, 'learning_rate': 6.2325581395348845e-06, 'epoch': 1.38}


 75%|███████▌  | 488/650 [26:41<08:05,  3.00s/it]

In [ ]:
# from transformers import TextStreamer

In [111]:
# test_idx = 56

# print(dataset_test[test_idx]["user_prompt"])

# messages = [
#     [
#         {'role': 'system','content' : SYSTEM_PROMPT},
#         {"role" : 'user', 'content' : dataset_test[test_idx]["user_prompt"]}
#     ]
# ]
# text = tokenizer.apply_chat_template(
#     messages[0],
#     tokenize = False,
#     add_generation_prompt = True, # Must add for generation
# ).removeprefix('<bos>')

Лет 5 назад был положительный опыт кредитования здесь.
Теперь же взял дебетовую карту на зарплатный проект. Все оформил онлайн. Карту привезли быстро, работает четко. Вопросов нет.


In [135]:
# _ = model.generate(
#     **tokenizer(text, return_tensors = "pt").to("cuda"),
#     max_new_tokens = 125,
#     temperature = 0.5, top_p = 0.95, top_k = 64,
#     streamer = TextStreamer(tokenizer, skip_prompt = True),
# )

# print(dataset_test[test_idx]["conversations"][-1]["content"])

[{'topic': 'Кредиты наличными', 'sentiment': 'positive'}, {'topic': 'Дебетовые карты', 'sentiment': 'positive'}]<end_of_turn>
[{'topic': 'Дебетовые карты', 'sentiment': 'positive'}, {'topic': 'Дистанционное обслуживание', 'sentiment': 'positive'}]


In [ ]:
model.save_pretrained("models/gemma-3-270m-it-revews-fine-tune-v4")  # Local saving
tokenizer.save_pretrained("models/gemma-3-270m-it-revews-fine-tune-v4")

In [ ]:
model.push_to_hub("JosephThePatrician/gemma3-270m-it-reviews-v4", tokenizer, token = "token")

In [ ]:
tokenizer.push_to_hub("JosephThePatrician/gemma3-270m-it-reviews-v4", token = "token")

In [ ]:
model.push_to_hub_merged("JosephThePatrician/gemma3-270m-it-reviews-v3", tokenizer, save_method = "merged_16bit", token = "token")

In [ ]:
model.push_to_hub("JosephThePatrician/gemma3-270m-it-reviews-v3", tokenizer, save_method = "merged_16bit", token = "token")

## Running on the whole dataset

In [29]:
from unsloth import FastLanguageModel
import torch
from itertools import islice
from math import floor

from tqdm.autonotebook import tqdm

In [30]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "mount/models/gemma-3-270m-it-revews-fine-tune-v1", # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length = 2048,
    load_in_4bit = False,
    load_in_8bit = True
)

==((====))==  Unsloth 2025.8.10: Fast Gemma3 patching. Transformers: 4.55.4. vLLM: 0.9.2.
   \\   /|    NVIDIA GeForce RTX 3050 Laptop GPU. Num GPUs = 1. Max memory: 4.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
!ls .cache/huggingface/hub

blobs  refs  snapshots


In [ ]:
# model.save_pretrained_merged("gemma3-270m-it-reviews-v1", tokenizer, save_method = "merged_16bit",)
model.push_to_hub_merged("JosephThePatrician/gemma3-270m-it-reviews-v1", tokenizer, save_method = "merged_16bit", token = "token")

In [ ]:
# model.push_to_hub_merged("JosephThePatrician/gemma3-270m-it-reviews-4bit-v1", tokenizer, save_method = "merged_4bit", token = "token")

In [7]:
model = FastLanguageModel.for_inference(model)

In [18]:
def batched(iterable, n):
    "Batch data into lists of length n. The last batch may be shorter."
    # batched('ABCDEFG', 3) --> ABC DEF G
    it = iter(iterable)
    while True:
        batch = tuple(islice(it, n))
        if not batch:
            return
        yield batch


def make_data_batch(reviews):
    messages = [
        [
            {'role': 'system','content' : SYSTEM_PROMPT},
            {"role" : 'user', 'content' : review}
        ]
        for review in reviews
    ]
    
    texts = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = True, # Must add for generation
    )
    
    texts = [text.removeprefix('<bos>') for text in texts]
    
    return texts


def predict_batch(reviews, batch_size=2):
    answers = []
    for review_batch in tqdm(batched(reviews, batch_size), total=floor(len(reviews) / batch_size)):
        
        # print(len(review_batch))
        # print(review_batch)
        
        texts = make_data_batch(review_batch)
        
        # print(len(texts))
        # print(texts)
        
        tokens = tokenizer(texts, padding_side="left", padding=True, return_tensors = "pt").to("cuda")
        
        out_tokens = model.generate(
            **tokens,
            max_new_tokens = 256,
            temperature = 0.1, top_p = 0.95, top_k = 64,
            # streamer = TextStreamer(tokenizer, skip_prompt = True),
        )
        
        answer_batch = tokenizer.batch_decode(out_tokens, skip_special_tokens=True)
        answer_batch = [answer_batch[i].split("\nmodel\n")[1] for i in range(len(answer_batch))]
        answers.extend(answer_batch)
    
    return answers

In [13]:
reviews = df["review_text"].values[:50]

In [16]:
predicted_topic_sentiment_pairs = predict_batch(reviews, 12)

  0%|          | 0/4 [00:00<?, ?it/s]

In [12]:
len(predicted_topic_sentiment_pairs)

4778

In [29]:
df.iloc[200]

review_id                                                 948975
date                                                  2025-04-17
review_text    11.04.25Г примерно в 12:30, я, пришёл в ТЦ «Тр...
topic                                               Обслуживание
subtopic                                                     NaN
sentiment                                               Negative
Name: 200, dtype: object

In [32]:
df["review_text"].values[100]

'Отрицательный опыт общения. Полное разочарование. Никому не посоветую открывать вклад в этом банке. Отвратительное обслуживание.'

In [33]:
predicted_topic_sentiment_pairs[100]

"[{'topic': 'Вклады', 'sentiment': 'negative'}, {'topic': 'Офисное обслуживание', 'sentiment': 'negative'}]"

In [16]:
with open("mount/data/predictions_v1.txt", mode="w") as f:
    f.write(str(predicted_topic_sentiment_pairs))

In [42]:
df_structured = pd.read_csv("mount/data/reviews_stuctured.csv")
df_structured

,reviewId,idSiteSpecific,source,date,review_text,topic,subtopic,rating
0,0,1000087,sravni.ru,2025-09-19,Вклад «Новые деньги» невозможно оформить без п...,Вклады,NaN,NaN
1,1,999494,sravni.ru,2025-09-18,В июне 2025 года я порекомендовал премиальную ...,Дебетовые карты,NaN,NaN
2,2,999142,sravni.ru,2025-09-17,Мошенниччиские аперации в интересах Ренессанс ...,Обслуживание,NaN,NaN
3,3,998360,sravni.ru,2025-09-15,Купил услугу Газпром Бонус «Премиум» за 2 990 ...,Дебетовые карты,NaN,NaN
4,4,998516,sravni.ru,2025-09-15,Производил оформление открытия срочного банков...,Вклады,«Накопительный»,NaN
...,...,...,...,...,...,...,...,...
4773,4773,7470,sravni.ru,2011-04-07,Ужастное обслуживание! Мало того потеряли доку...,Обслуживание,NaN,NaN
4774,4774,7049,sravni.ru,2011-03-28,Могут заблокировать рассчетную или кредитную к...,Кредитные карты,NaN,NaN
4775,4775,5221,sravni.ru,2011-01-25,"Мало того уже прошла неделя, а ПТС так и не ве...",Автокредиты,NaN,NaN
4776,4776,5053,sravni.ru,2011-01-16,Газпромбанк– отличный банк с отличными сотрудн...,Ипотека,NaN,NaN


In [43]:
df_review_topics = pd.read_csv("mount/data/reviews_topics.csv")
df_review_topics

,id,reviewId,topicId,sentiment
0,0,0,0,Negative
1,1,1,1,Negative
2,2,2,0,Negative
3,3,3,17,Negative
4,4,4,17,Negative
...,...,...,...,...
4773,4773,4773,2,Negative
4774,4774,4774,4,Negative
4775,4775,4775,8,Negative
4776,4776,4776,17,Positive


In [64]:
df_topics_info = pd.read_csv("mount/data/topics_info.csv")
df_topics_info

,id,name,description
0,0,Вклады,NaN
1,1,Дебетовые карты,NaN
2,2,Обслуживание,NaN
3,3,Дистанционное обслуживание,NaN
4,4,Кредитные карты,NaN
5,5,Кредиты наличными,NaN
6,6,Другие услуги,NaN
7,7,Обмен валют,NaN
8,8,Ипотека,NaN
9,9,Автокредиты,NaN


In [40]:
predicted_topic_sentiment_pairs[0]

"[{'topic': 'Вклады', 'sentiment': 'negative'}, {'topic': 'Страховые и сервисные продукты', 'sentiment': 'negative'}]"

In [56]:
ids = []
topics = []
sentiments = []

for i, pairs_str in enumerate(predicted_topic_sentiment_pairs):
    # topics_sentiments_pairs = review["topic_sentiment_pairs"]
    review_id = df_structured["reviewId"][i]
    # print(pairs_str)
    pairs = eval(pairs_str)
    for pair in pairs:
        ids.append(int(review_id))
        topics.append(pair["topic"])
        sentiments.append(pair["sentiment"])

In [27]:
print("123456\r789")

789456


In [ ]:
pd.to_datetime(df["date"]).val

4777   2011-01-10
4776   2011-01-16
4775   2011-01-25
4774   2011-03-28
4773   2011-04-07
          ...    
4      2025-09-15
3      2025-09-15
2      2025-09-17
1      2025-09-18
0      2025-09-19
Name: date, Length: 4778, dtype: datetime64[ns]

In [75]:
unique_topics_sorted = pd.Series(topics).value_counts().index.values

unique_topics_ids = list(range(len(unique_topics_sorted)))

topics_ids_dict = {
    unique_topics_sorted[i] : unique_topics_ids[i] for i in range(len(unique_topics_sorted))
}

df_topics_info_v1 = pd.DataFrame(
    {
        "id" : unique_topics_ids,
        "name" : unique_topics_sorted,
        "description" : None
    }
)

# df_topics_info_v1

In [77]:
df_review_topics

,id,reviewId,topicId,sentiment
0,0,0,0,Negative
1,1,1,1,Negative
2,2,2,0,Negative
3,3,3,17,Negative
4,4,4,17,Negative
...,...,...,...,...
4773,4773,4773,2,Negative
4774,4774,4774,4,Negative
4775,4775,4775,8,Negative
4776,4776,4776,17,Positive


In [86]:
topicids = [topics_ids_dict[topics[i]] for i in range(len(topics))]

In [90]:
df_topics_sentiments_v1 = pd.DataFrame(
    {
        "id" : list(range(len(ids))),
        "reviewId" : ids,
        "topicId" : topicids,
        "sentiment" : sentiments
    }
)

df_topics_sentiments_v1

,id,reviewId,topicId,sentiment
0,0,0,6,negative
1,1,0,7,negative
2,2,1,0,negative
3,3,1,17,negative
4,4,1,1,negative
...,...,...,...,...
8948,8948,4775,0,negative
8949,8949,4776,4,positive
8950,8950,4776,2,positive
8951,8951,4777,3,negative


In [91]:
df_topics_info_v1.to_csv("mount/data/structured_data/topics_info.csv", index=False)

df_topics_sentiments_v1.to_csv("mount/data/structured_data/reviews_topics_v1.csv", index=False)